# Fine-tuning Gemma 1b

On the raw conversations dataset.

In [2]:
dataset_path = "../../output/dataset/dspy-dataset.csv"

In [3]:
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    dtype=None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 07-23 21:29:39 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.7.8: Fast Gemma3 patching. Transformers: 4.53.3. vLLM: 0.8.3.
   \\   /|    NVIDIA RTX 2000 Ada Generation Laptop GPU. Num GPUs = 1. Max memory: 7.747 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"], 
    use_rslora=True,
    use_gradient_checkpointing="unsloth"  # type: ignore[assignment]
)

Unsloth: Making `model.base_model.model.model` require gradients


In [5]:
import pandas as pd
import json
import ast
from datasets import Dataset


# TODO: Fix the dataset itself, it has single quotes in the JSON strings which
# is not valid JSON.
def load_dataset():
    messages = []

    df = pd.read_csv(dataset_path, sep=",", encoding="utf-8")
    for i, row in df.iterrows():
        try:
            # Use ast.literal_eval to handle single-quoted JSON strings
            conversation = ast.literal_eval(row['conversations'])
            messages.append(conversation)
        except (ValueError, SyntaxError) as e:
            # Fallback: try to fix single quotes and parse as JSON
            try:
                conversations_str = row['conversations'].replace("'", '"')
                conversation = json.loads(conversations_str)
                messages.append(conversation)
            except json.JSONDecodeError as json_e:
                raise ValueError(f"Error decoding conversations for row {i}: {e}, also tried JSON fix: {json_e}")

    res = Dataset.from_dict({"messages": messages})
    return res

dataset = load_dataset()
dataset

Dataset({
    features: ['messages'],
    num_rows: 244
})